# IFRS 9 Expected Credit Loss (ECL) Model
## Notebook 4 — ECL Calculation & Results

**ECL Formula:**
$$ECL = PD \times LGD \times EAD \times DF$$

Where DF = Discount Factor using Effective Interest Rate (EIR)

**Probability-weighted scenarios (IFRS 9.B5.5.41):**
| Scenario | Macro Scalar | Weight |
|----------|-------------|--------|
| Benign   | ×0.70       | 30%    |
| Base     | ×1.00       | 50%    |
| Adverse  | ×1.80       | 20%    |

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import load_loan_portfolio, assign_stage
from src.pd_model import PDModel
from src.lgd_ead import compute_ead, compute_lgd
from src.ecl_calculator import ECLCalculator

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

# Build full pipeline
df = assign_stage(load_loan_portfolio())
pd_model = PDModel(macro_scalar=1.0)
df = pd_model.assign_pd(df)
df['ead'] = compute_ead(df)
df['lgd'] = compute_lgd(df, collateral_haircut=0.20)

calculator = ECLCalculator(discount_rate=0.05)
df = calculator.compute_ecl(df)

total_ead = df['ead'].sum()
total_ecl = df['ecl'].sum()
print(f'Total EAD: THB {total_ead/1e9:.3f}B')
print(f'Total ECL: THB {total_ecl/1e6:.1f}M  (Coverage: {total_ecl/total_ead:.2%})')

## 1. ECL Summary by IFRS 9 Stage

In [ ]:
summary = calculator.ecl_summary(df)
print('=== IFRS 9 ECL Summary ===')
print(summary[['n_loans','total_ead','total_ecl','avg_pd','avg_lgd',
               'ecl_pct_of_ead','pct_of_portfolio']].to_string())

print(f'\nProvision Coverage Ratio (Total): {total_ecl/total_ead:.2%}')

## 2. ECL Dashboard

In [ ]:
fig = calculator.plot_ecl_dashboard(df)
plt.savefig('../plots/04_ecl_dashboard.png', bbox_inches='tight')
plt.show()

## 3. ECL by Sector

In [ ]:
sector_ecl = (
    df.groupby('sector')
    .agg(total_ead=('ead','sum'), total_ecl=('ecl','sum'),
         avg_coverage=('coverage_ratio','mean'))
    .assign(coverage_pct=lambda x: (x['total_ecl']/x['total_ead']*100).round(2))
    .sort_values('total_ecl', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

(sector_ecl['total_ecl'] / 1e6).plot(kind='bar', ax=axes[0], color='#e74c3c', alpha=0.85)
axes[0].set_title('Total ECL by Sector (THB M)')
axes[0].set_ylabel('THB Million')
axes[0].tick_params(axis='x', rotation=30)

sector_ecl['coverage_pct'].plot(kind='bar', ax=axes[1], color='#e67e22', alpha=0.85)
axes[1].set_title('ECL Coverage Ratio by Sector (%)')
axes[1].set_ylabel('Coverage Ratio (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Sector-Level ECL Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/04_ecl_by_sector.png', bbox_inches='tight')
plt.show()

print(sector_ecl.to_string())

## 4. Scenario Comparison

In [ ]:
scenarios = {'Benign (×0.7)':0.70, 'Base (×1.0)':1.00, 'Adverse (×1.8)':1.80}
results = []
for name, scalar in scenarios.items():
    m = PDModel(macro_scalar=scalar)
    df_s = m.assign_pd(df.copy())
    df_s = ECLCalculator(discount_rate=0.05).compute_ecl(df_s)
    results.append({'scenario': name,
                    'total_ecl_thb_m': df_s['ecl'].sum() / 1e6,
                    'coverage_pct': df_s['ecl'].sum() / df_s['ead'].sum() * 100})

scenario_df = pd.DataFrame(results)
print('=== ECL by Macro Scenario ===')
print(scenario_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors_sc = ['#27ae60','#2980b9','#e74c3c']
bars = ax.bar(scenario_df['scenario'], scenario_df['total_ecl_thb_m'],
              color=colors_sc, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, scenario_df['total_ecl_thb_m']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'THB {val:.0f}M', ha='center', fontsize=10)
ax.set_title('Total ECL by Macro Scenario', fontweight='bold')
ax.set_ylabel('ECL (THB Million)')
plt.tight_layout()
plt.savefig('../plots/04_ecl_scenario.png', bbox_inches='tight')
plt.show()